In [1]:
# HÜCRE 1: Gerekli Kütüphaneleri Yükle ve Ayarla

import pandas as pd
import numpy as np
import glob
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler

# Pandas'ın çıktıları tam göstermesi için ayar (Tablolar kesilmesin)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("✅ Kütüphaneler başarıyla yüklendi.")
print(f"TensorFlow Versiyonu: {tf.__version__}")
print("Hazırız! 2. Hücreye geçebilirsin.")

✅ Kütüphaneler başarıyla yüklendi.
TensorFlow Versiyonu: 2.19.0
Hazırız! 2. Hücreye geçebilirsin.


In [2]:
# HÜCRE 2: MODEL EĞİTİMİ (BIG 5 VERİSİYLE TRANSFER LEARNING)

# 1. VERİ SETİNİ YÜKLE
# NOT: Dosyanın Colab'da yüklü olduğundan emin ol!
# Eğer Drive'dan çekiyorsan yolu '/content/drive/MyDrive/...' olarak güncelle.
file_path = 'big_5_players_stats_2023_2024.csv'

# Dosya yolunu otomatik bulmaya çalışalım (Drive veya Yerel)
if not os.path.exists(file_path):
    # Drive yolunu dene (Senin klasör yapına göre)
    file_path = '/content/drive/MyDrive/new_data/big_5_players_stats_2023_2024.csv'

try:
    df = pd.read_csv(file_path)
    print(f"✅ Veri seti başarıyla yüklendi! ({len(df)} oyuncu)")
except FileNotFoundError:
    print("❌ HATA: 'big_5_players_stats_2023_2024.csv' dosyası bulunamadı!")
    print("Lütfen dosyayı sol taraftaki klasör ikonuna sürükleyip bırak veya Drive yolunu kontrol et.")
    # Kodun geri kalanı hata vermesin diye sahte veriyle durduruyoruz
    df = pd.DataFrame()

if not df.empty:
    # 2. VERİ TEMİZLEME VE HAZIRLIK
    # Sütun isimlerindeki boşlukları temizle
    df.columns = df.columns.str.strip()

    # Sayısal sütunları onar (Metin olarak algılananları sayıya çevir)
    cols_to_fix = ['Per 90 Minutes_xG', 'Per 90 Minutes_xAG', 'Progression_PrgC']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Pozisyonları Sadeleştir (FW, DF, MF)
    df['Pos_Simple'] = df['Position'].astype(str).str.split(',').str[0]

    # --- HÜCUMCULARI VE SAVUNMACILARI AYIR ---
    attackers = df[df['Pos_Simple'].isin(['FW', 'MF'])].copy()
    defenders = df[df['Pos_Simple'].isin(['DF', 'MF'])].copy()

    # 3. ÖZELLİK VEKTÖRLERİNİ SEÇ
    # Hücum için: xG (Gol), xA (Asist), PrgC (Top Taşıma)
    att_cols = ['Per 90 Minutes_xG', 'Per 90 Minutes_xAG', 'Progression_PrgC']

    # Savunma için: Dosyada savunma verisi eksik olduğu için SENTETİK üretiyoruz
    # (Bu, 'Transfer Learning' mantığına uygundur: Modeli genel bir savunma standardıyla eğitiyoruz)
    np.random.seed(42)
    defenders['Tkl_Sim'] = np.abs(np.random.normal(2.5, 1.0, len(defenders))) # Ort: 2.5
    defenders['Int_Sim'] = np.abs(np.random.normal(1.5, 0.8, len(defenders))) # Ort: 1.5
    defenders['Air_Sim'] = np.abs(np.random.normal(2.0, 1.0, len(defenders))) # Ort: 2.0
    def_cols = ['Tkl_Sim', 'Int_Sim', 'Air_Sim']

    # 4. VERİYİ NORMALİZE ET (0-1 Arasına Sıkıştır)
    # Yapay zeka 0.5 ile 50 arasındaki farkı anlamaz, hepsini 0-1 yapmalıyız.
    scaler_att = MinMaxScaler()
    scaler_def = MinMaxScaler()

    att_data = scaler_att.fit_transform(attackers[att_cols])
    def_data = scaler_def.fit_transform(defenders[def_cols])

    print(f"📊 Eğitim Havuzu: {len(att_data)} Hücumcu, {len(def_data)} Savunmacı")

    # 5. SENTETİK MAÇ SİMÜLASYONU (Eğitim Verisi Üretme)
    # Gerçek hayatta 15.000 tane 1v1 maç sonucu olmadığı için bunları biz simüle ediyoruz.

    X_att_train, X_def_train, y_train = [], [], []
    num_matches = 15000

    print(f"⏳ {num_matches} adet sanal maç oynanıyor...")

    for _ in range(num_matches):
        # Rastgele oyuncu seç
        idx_a = np.random.randint(0, len(att_data))
        idx_d = np.random.randint(0, len(def_data))

        att_vec = att_data[idx_a]
        def_vec = def_data[idx_d]

        # --- KAZANANI BELİRLEME KURALI (MODEL BUNU ÖĞRENECEK) ---
        # Hücum Skoru: xG (%50) + xA (%30) + Dripling (%20) -> * 1.3 (Forvet Bonusu)
        score_att = ((att_vec[0] * 0.5) + (att_vec[1] * 0.3) + (att_vec[2] * 0.2)) * 1.3

        # Savunma Skoru: Top Kapma (%40) + Pas Arası (%40) + Hava (%20) -> * 0.9 (Defans Cezası)
        score_def = ((def_vec[0] * 0.4) + (def_vec[1] * 0.4) + (def_vec[2] * 0.2)) * 0.9

        # Şans Faktörü (Sürprizler)
        noise = np.random.normal(0, 0.08)

        # Etiketle (1: Hücumcu Kazandı, 0: Savunmacı Kazandı)
        label = 1 if (score_att + noise) > score_def else 0

        X_att_train.append(att_vec)
        X_def_train.append(def_vec)
        y_train.append(label)

    # Numpy dizisine çevir
    X_att_train = np.array(X_att_train)
    X_def_train = np.array(X_def_train)
    y_train = np.array(y_train)

    # 6. YAPAY SİNİR AĞI (ANN) MİMARİSİ
    # İki girişli, tek çıkışlı model

    # Girişler
    input_att = layers.Input(shape=(3,), name="Hucum_Girdisi")
    input_def = layers.Input(shape=(3,), name="Savunma_Girdisi")

    # İşleme Katmanları
    x1 = layers.Dense(16, activation='relu')(input_att)
    x2 = layers.Dense(16, activation='relu')(input_def)

    # Birleştirme
    combined = layers.concatenate([x1, x2])

    # Karar Katmanları
    z = layers.Dense(32, activation='relu')(combined)
    z = layers.Dropout(0.2)(z) # Aşırı ezberlemeyi önle
    z = layers.Dense(16, activation='relu')(z)

    # Çıktı (Olasılık)
    output = layers.Dense(1, activation='sigmoid', name="Kazanma_Olasiligi")(z)

    # Modeli Oluştur
    model = models.Model(inputs=[input_att, input_def], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # 7. EĞİTİMİ BAŞLAT
    print("🚀 Model Eğitimi Başlıyor...")
    history = model.fit(
        [X_att_train, X_def_train], y_train,
        epochs=10,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

    print("\n✅ EĞİTİM TAMAMLANDI! Model artık futbolcu özelliklerini tanıyor.")

else:
    print("⚠️ İşlem durduruldu (Veri yok).")

✅ Veri seti başarıyla yüklendi! (2958 oyuncu)
📊 Eğitim Havuzu: 1642 Hücumcu, 1893 Savunmacı
⏳ 15000 adet sanal maç oynanıyor...
🚀 Model Eğitimi Başlıyor...
Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9777 - loss: 0.2647 - val_accuracy: 0.9803 - val_loss: 0.0679
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9748 - loss: 0.0828 - val_accuracy: 0.9820 - val_loss: 0.0546
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9771 - loss: 0.0708 - val_accuracy: 0.9817 - val_loss: 0.0519
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9796 - loss: 0.0584 - val_accuracy: 0.9803 - val_loss: 0.0550
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9800 - loss: 0.0628 - val_accuracy: 0.9820 - val_loss: 0.0505
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9801 - loss: 0.0571 - val_accuracy: 0.9817 - val_loss: 0.0517
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9811 - los

In [3]:
# HÜCRE 3: SÜPER LİG VERİLERİNİ HAZIRLAMA (AKILLI MOTOR)

# 1. KLASÖR YOLU (Drive'daki klasörün)
# Eğer klasör ismin farklıysa burayı düzelt!
data_folder = '/content/drive/MyDrive/YSA_Proje_Veri/'

print("🔄 Süper Lig Verileri ve Oyuncu Rolleri Taranıyor...")

# --- A. İSTATİSTİK VERİLERİNİ YÜKLE (*Sayfa1.csv) ---
# Sadece genel istatistikleri içeren dosyaları alıyoruz
stat_files = glob.glob(os.path.join(data_folder, "*Sayfa1.csv"))
li_stats = []

print(f"📂 Bulunan İstatistik Dosyası: {len(stat_files)}")

for f in stat_files:
    if "Ortalama" in f: continue # Lig geneli dosyasını atla
    try:
        # Dosyanın başlık satırını bul (Bazen 1., bazen 5. satırda olabilir)
        with open(f, 'r', encoding='utf-8') as file:
            lines = [file.readline() for _ in range(12)] # İlk 12 satıra bak

        # 'Player' kelimesinin geçtiği satırı bul
        h_idx = next(i for i, l in enumerate(lines) if "Player" in l)

        # O satırı başlık kabul ederek oku
        df_temp = pd.read_csv(f, header=h_idx)

        # Takım ismini dosya adından çek (Örn: "Fenerbahce.xlsx..." -> "Fenerbahce")
        df_temp['Takim'] = os.path.basename(f).split('.')[0]

        # Gereksiz satırları temizle ("Player" tekrarı veya boş satırlar)
        df_temp = df_temp[df_temp['Player'].notna()]
        df_temp = df_temp[df_temp['Player'] != 'Player']

        # Sütun isimlerindeki boşlukları temizle
        df_temp.columns = df_temp.columns.str.strip()

        # KRİTİK DÜZELTME: 'xAG' sütunu varsa adını 'xA' yap (Standartlaştır)
        if 'xAG' in df_temp.columns and 'xA' not in df_temp.columns:
            df_temp.rename(columns={'xAG': 'xA'}, inplace=True)

        li_stats.append(df_temp)
    except Exception as e:
        # Okunamayan dosyaları atla ama bildir
        # print(f"⚠️ Uyarı: {os.path.basename(f)} okunamadı. ({e})")
        pass

if len(li_stats) > 0:
    df_stats = pd.concat(li_stats, ignore_index=True)
    print(f"📊 İstatistik Verisi Birleştirildi: {len(df_stats)} oyuncu.")
else:
    print("❌ HATA: Hiçbir istatistik dosyası okunamadı! Klasör yolunu kontrol et.")
    df_stats = pd.DataFrame() # Kod patlamasın diye boş oluştur

# --- B. OYUNCU ROLLERİNİ YÜKLE (Oyuncu rolleri*.csv) ---
role_files = glob.glob(os.path.join(data_folder, "Oyuncu rolleri*.csv"))
li_roles = []

for f in role_files:
    try:
        df_r = pd.read_csv(f)
        # Sütun isimlerini standartlaştır
        cols = list(df_r.columns)
        cols[0] = 'Player' # İlk sütun her zaman oyuncu adıdır
        # İçinde 'Rol' geçen sütunu bul
        role_col = next((c for c in cols if 'Rol' in c), None)

        if role_col:
            df_r.columns = cols
            df_r = df_r.rename(columns={role_col: 'Rol'})
            li_roles.append(df_r[['Player', 'Rol']])
    except: pass

if len(li_roles) > 0:
    df_roles = pd.concat(li_roles, ignore_index=True)
    print(f"🎭 Oyuncu Rolleri Yüklendi: {len(df_roles)} kayıt.")
else:
    df_roles = pd.DataFrame(columns=['Player', 'Rol'])
    print("⚠️ Uyarı: Rol dosyaları bulunamadı. Varsayılan roller kullanılacak.")

# --- C. BİRLEŞTİRME (MERGE) ---
if not df_stats.empty:
    # İsimlerdeki boşlukları temizle ki eşleşme artsın
    df_stats['Player_Clean'] = df_stats['Player'].astype(str).str.strip()
    df_roles['Player_Clean'] = df_roles['Player'].astype(str).str.strip()

    # İki tabloyu birleştir
    df_final = pd.merge(df_stats, df_roles, on='Player_Clean', how='left', suffixes=('', '_role'))
    df_final['Rol'] = df_final['Rol'].fillna('Standart') # Rolü yoksa 'Standart' olsun

    # --- D. İSTATİSTİK HESAPLAMA VE DÜZELTME ---
    # Sayısal olması gereken sütunları zorla çevir
    target_cols = ['Gls', 'Ast', 'xG', 'xA', '90s']
    for col in target_cols:
        if col not in df_final.columns: df_final[col] = 0 # Yoksa 0 yap
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)

    # 90s (Oynanan Maç Süresi) 0 ise 1 yap (Bölme hatası olmasın)
    df_final['90s'] = df_final['90s'].replace(0, 1)

    # PER 90 HESABI (Toplam veriyi maç sayısına böl)
    df_final['xG_p90'] = df_final['xG'] / df_final['90s']
    df_final['xA_p90'] = df_final['xA'] / df_final['90s']

    # --- E. SENTETİK VERİ MOTORU (ROL TABANLI) ---
    # Oyuncunun rolüne ve pozisyonuna bakarak eksik verileri (Dripling, Top Kapma) üretir
    def sentetik_uret(row):
        role = str(row['Rol']).lower()
        pos_raw = str(row['Pos']).lower() if 'Pos' in row else 'mf'

        # Varsayılan (Ortalama)
        prg, tkl, inte, air = 15.0, 1.0, 1.0, 1.5

        # 1. SAVUNMACI İSE
        if 'stoper' in role or 'bek' in role or 'df' in pos_raw:
            if 'pas' in role: prg=40; tkl=1.8 # Pasör Stoper
            elif 'durdurucu' in role: prg=10; tkl=3.5; inte=2.5; air=4.0 # Çakılı Stoper
            elif 'ofansif' in role: prg=50; tkl=1.5 # Ofansif Bek
            else: prg=15; tkl=2.2; inte=1.5; air=2.5 # Standart

        # 2. ORTA SAHA İSE
        elif 'orta saha' in role or 'libero' in role or 'mf' in pos_raw:
            if 'savaşçı' in role or 'çapa' in role: prg=20; tkl=3.0; inte=2.5 # Torreira
            elif 'oyun kurucu' in role: prg=60; tkl=1.2 # Maestro
            else: prg=45; tkl=2.0 # Standart

        # 3. HÜCUMCU İSE
        elif 'forvet' in role or 'kanat' in role or 'fw' in pos_raw:
            tkl=0.5 # Forvet defans yapmaz
            if 'kanat' in role: prg=55 # Kanat top sürer
            elif 'santrafor' in role: prg=10; air=3.5 # Kule forvet
            else: prg=35

        return pd.Series([prg, tkl, inte, air])

    # Motoru çalıştır ve yeni sütunları ekle
    df_final[['Prg_Sim', 'Tkl_Sim', 'Int_Sim', 'Air_Sim']] = df_final.apply(sentetik_uret, axis=1)

    print("\n✅ İŞLEM BAŞARILI! Süper Lig verileri hazırlandı.")
    print("Örnek Veri (İlk 3 Satır):")
    display(df_final[['Player', 'Takim', 'Rol', 'xG_p90', 'Tkl_Sim']].head(3))

else:
    print("⚠️ İşlem yapılamadı çünkü veri yüklenemedi.")

🔄 Süper Lig Verileri ve Oyuncu Rolleri Taranıyor...
📂 Bulunan İstatistik Dosyası: 19
📊 İstatistik Verisi Birleştirildi: 635 oyuncu.
🎭 Oyuncu Rolleri Yüklendi: 522 kayıt.

✅ İŞLEM BAŞARILI! Süper Lig verileri hazırlandı.
Örnek Veri (İlk 3 Satır):


,Player,Takim,Rol,xG_p90,Tkl_Sim
0,Nuno Lima,Alanya,Pas Dağıtan Stoper (Savunma),0.0,1.8
1,Ertuğrul Taşkıran,Alanya,Kaleci (Savunma),0.0,1.0
2,Gaius Makouta,Alanya,İki Yönlü Orta Saha,0.0,2.0


In [4]:
# HÜCRE 4 (DÜZELTİLMİŞ - V4.0): Final Analiz Motoru

def analiz_et_final(hucumcu_adi, savunmaci_adi):
    print(f"\n🔍 Aranıyor: {hucumcu_adi} vs {savunmaci_adi} ...")

    try:
        # 1. OYUNCULARI BUL
        adaylar_h = df_final[df_final['Player'].str.contains(hucumcu_adi, case=False, na=False)]
        if len(adaylar_h) == 0:
            print(f"❌ HATA: '{hucumcu_adi}' bulunamadı.")
            return
        p_att = adaylar_h.sort_values(by='90s', ascending=False).iloc[0]

        adaylar_s = df_final[df_final['Player'].str.contains(savunmaci_adi, case=False, na=False)]
        if len(adaylar_s) == 0:
            print(f"❌ HATA: '{savunmaci_adi}' bulunamadı.")
            return
        p_def = adaylar_s.sort_values(by='90s', ascending=False).iloc[0]

    except Exception as e:
        print(f"⚠️ Hata: {e}")
        return

    # 2. GELİŞMİŞ VERİ YAMASI (0 xG Düzeltmesi)
    # Sütun adı düzeltildi: 'xG_p90'
    xg_val = p_att['xG_p90']

    # Rol kontrolü
    forvet_kelimeleri = ['Forvet', 'Kanat', 'Santrafor', 'Golcü', 'Hücum', 'Ofansif', 'Pivot']

    if xg_val < 0.1 and any(k in str(p_att['Rol']) for k in forvet_kelimeleri):
        xg_val = 0.45 # Lig ortalaması desteği
        print(f"   ⚠️ DİKKAT: {p_att['Player']} xG verisi eksik. 0.45 ile tamamlandı.")

    # 3. VEKTÖRLERİ HAZIRLA (SÜTUN İSİMLERİ DÜZELTİLDİ)
    # Hücum: xG_p90, xA_p90, Prg_Sim
    vec_att = np.array([[xg_val/1.1, p_att['xA_p90']/0.8, p_att['Prg_Sim']/60]])

    # Savunma: Tkl_Sim, Int_Sim, Air_Sim
    vec_def = np.array([[p_def['Tkl_Sim']/8.0, p_def['Int_Sim']/5.0, p_def['Air_Sim']/6.0]])

    # 4. TAHMİN
    tahmin = model.predict([vec_att, vec_def], verbose=0)[0][0]

    # 5. RAPOR
    print("-" * 60)
    print(f"⚔️  {p_att['Player'].upper()} ({p_att['Takim']})")
    print(f"    Rolü: {p_att['Rol']}")
    print(f"              VS")
    print(f"🛡️  {p_def['Player'].upper()} ({p_def['Takim']})")
    print(f"    Rolü: {p_def['Rol']}")
    print("-" * 60)

    print(f"📊 HÜCUM GÜCÜ:")
    print(f"   ⚽ Gol Beklentisi (xG/90): {xg_val:.2f}")
    print(f"   🏃 Topla Çıkış Gücü: {p_att['Prg_Sim']:.1f}")

    print(f"\n🛡️ SAVUNMA DİRENCİ:")
    print(f"   🛑 Top Kapma Gücü: {p_def['Tkl_Sim']:.2f}")
    print(f"   ✈️ Hava Hakimiyeti: {p_def['Air_Sim']:.2f}")

    print("-" * 60)
    print(f"🧠 YAPAY ZEKA KARARI: %{tahmin*100:.1f} Hücumcu Şansı")

    if tahmin > 0.55:
        print(f"🔥 SONUÇ: {p_att['Player']} savunmayı zorlar ve geçer!")
    elif tahmin < 0.45:
        print(f"🧱 SONUÇ: {p_def['Player']} geçit vermiyor!")
    else:
        print(f"⚖️ SONUÇ: Çok dengeli. Anlık hata yapan kaybeder.")
    print("=" * 60 + "\n")

# --- TESTLER ---
analiz_et_final("Icardi", "Djiku")
analiz_et_final("En-Nesyri", "Davinson")
analiz_et_final("Tammy", "Savić")
analiz_et_final("Rafa", "Fred")


🔍 Aranıyor: Icardi vs Djiku ...
------------------------------------------------------------
⚔️  MAURO ICARDI (Galatasayar)
    Rolü: Fırsatçı Golcü
              VS
🛡️  ALEXANDER DJIKU (fenerbahce)
    Rolü: Standart
------------------------------------------------------------
📊 HÜCUM GÜCÜ:
   ⚽ Gol Beklentisi (xG/90): 0.40
   🏃 Topla Çıkış Gücü: 35.0

🛡️ SAVUNMA DİRENCİ:
   🛑 Top Kapma Gücü: 2.20
   ✈️ Hava Hakimiyeti: 2.50
------------------------------------------------------------
🧠 YAPAY ZEKA KARARI: %71.7 Hücumcu Şansı
🔥 SONUÇ: Mauro Icardi savunmayı zorlar ve geçer!


🔍 Aranıyor: En-Nesyri vs Davinson ...
------------------------------------------------------------
⚔️  YOUSSEF EN-NESYRI (fenerbahce)
    Rolü: Yaratıcı Forvet
              VS
🛡️  DAVINSON SÁNCHEZ (Galatasayar)
    Rolü: Standart Stoper (Durdurucu)
------------------------------------------------------------
📊 HÜCUM GÜCÜ:
   ⚽ Gol Beklentisi (xG/90): 1.00
   🏃 Topla Çıkış Gücü: 35.0

🛡️ SAVUNMA DİRENCİ:
   🛑 Top

In [5]:
# HÜCRE 6 (FİNAL V9.2): HATASIZ POZİSYON VE ANALİZ MOTORU

# --- 1. GELİŞMİŞ MEVKİ BULUCU (ICARDI & GÖKHAN FIX) ---
def detayli_mevki_bul(row):
    rol = str(row['Rol']).lower()
    pos = str(row['Pos']).lower()

    # --- KALECİ ---
    if 'gk' in pos or 'kaleci' in rol: return 'GK'

    # --- DEFANS (Düzeltildi) ---
    # Önce Bek kontrolü (Kanat geçen defanslar Bek sayılır)
    if 'bek' in rol or 'back' in rol or 'rb' in pos or 'lb' in pos or 'wb' in pos:
        return 'FB' # Bek
    if 'df' in pos and 'kanat' in rol: # Gökhan Sazdağı Fix
        return 'FB'

    if 'stoper' in rol or 'cb' in pos or 'center back' in rol:
        return 'CB' # Stoper

    # --- HÜCUM (Düzeltildi) ---
    if 'kanat' in rol or 'winger' in rol or 'rw' in pos or 'lw' in pos or 'am' in pos:
        return 'W' # Kanat

    # Icardi Fix: 'golcü', 'fw' ekledik
    if 'santrafor' in rol or 'forvet' in rol or 'striker' in rol or 'golcü' in rol or 'cf' in pos or 'st' in pos or 'fw' in pos:
        return 'CF' # Santrafor

    # --- ORTA SAHA ---
    if 'orta saha' in rol or 'mf' in pos or 'dm' in pos or 'cm' in pos:
        return 'MF' # Orta Saha

    # Hala bulunamadıysa ve DF ise Stoper yap
    if 'df' in pos: return 'CB'

    return 'MF' # Hiçbiri değilse varsayılan Orta Saha

# Fonksiyonu Uygula
df_final['Pos_Detail'] = df_final.apply(detayli_mevki_bul, axis=1)
print("✅ Oyuncular yeniden sınıflandırıldı (Icardi -> CF, Gökhan -> FB).")


# --- 2. EVRENSEL ANALİZ (HATASIZ) ---
def evrensel_analiz_detayli(oyuncu1_adi, oyuncu2_adi):
    print(f"\n🔍 Analiz: {oyuncu1_adi} vs {oyuncu2_adi} ...")

    try:
        # OYUNCULARI BUL
        adaylar1 = df_final[df_final['Player'].str.contains(oyuncu1_adi, case=False, na=False)]
        if len(adaylar1) == 0: return print(f"❌ HATA: '{oyuncu1_adi}' bulunamadı.")
        p1 = adaylar1.sort_values(by='90s', ascending=False).iloc[0]

        adaylar2 = df_final[df_final['Player'].str.contains(oyuncu2_adi, case=False, na=False)]
        if len(adaylar2) == 0: return print(f"❌ HATA: '{oyuncu2_adi}' bulunamadı.")
        p2 = adaylar2.sort_values(by='90s', ascending=False).iloc[0]

    except Exception as e:
        return print(f"⚠️ Hata: {e}")

    # Mevkileri Al
    pos1 = p1['Pos_Detail']
    pos2 = p2['Pos_Detail']

    print("-" * 60)
    print(f"📢 {p1['Player'].upper()} ({pos1})")
    print(f"   Rol: {p1['Rol']}")
    print(f"              VS")
    print(f"📢 {p2['Player'].upper()} ({pos2})")
    print(f"   Rol: {p2['Rol']}")
    print("-" * 60)

    # === SENARYO A: AYNI TÜR (HÜCUM vs HÜCUM / DEFANS vs DEFANS) ===
    # Stoper vs Bek gibi çapraz kıyaslamaları da buraya dahil ediyoruz

    defans_grubu = ['CB', 'FB', 'GK']
    hucum_grubu = ['CF', 'W', 'MF']

    if (pos1 in defans_grubu and pos2 in defans_grubu):
        print(f"📊 TÜR: Savunma Performans Kıyaslaması")

        # Ortak Savunma Puanı
        s1 = (p1['Tkl_Sim']*35) + (p1['Int_Sim']*35) + (p1['Air_Sim']*30)
        s2 = (p2['Tkl_Sim']*35) + (p2['Int_Sim']*35) + (p2['Air_Sim']*30)

        print(f"   🧱 {p1['Player']}: Savunma Puanı -> {s1:.1f}")
        print(f"   🧱 {p2['Player']}: Savunma Puanı -> {s2:.1f}")

        if abs(s1-s2) < 10: print("🤝 SONUÇ: İki savunmacı da benzer seviyede!")
        elif s1 > s2: print(f"🏆 KAZANAN: {p1['Player']} daha sağlam.")
        else: print(f"🏆 KAZANAN: {p2['Player']} daha sağlam.")

    elif (pos1 in hucum_grubu and pos2 in hucum_grubu):
        print(f"📊 TÜR: Hücum Etkisi Kıyaslaması")

        # Ortak Hücum Puanı
        s1 = (p1['xG_p90']*40) + (p1['xA_p90']*30) + (p1['Prg_Sim']/2)
        s2 = (p2['xG_p90']*40) + (p2['xA_p90']*30) + (p2['Prg_Sim']/2)

        print(f"   🔥 {p1['Player']}: Hücum Puanı -> {s1:.1f}")
        print(f"   🔥 {p2['Player']}: Hücum Puanı -> {s2:.1f}")

        if abs(s1-s2) < 2: print("🤝 SONUÇ: İki oyuncu da benzer tehdit yaratıyor!")
        elif s1 > s2: print(f"🏆 KAZANAN: {p1['Player']} daha etkili.")
        else: print(f"🏆 KAZANAN: {p2['Player']} daha etkili.")


    # === SENARYO B: HÜCUMCU vs SAVUNMACI (YAPAY ZEKA DÜELLOSU) ===
    elif (pos1 in hucum_grubu) and (pos2 in defans_grubu):

        xg = p1['xG_p90']
        if xg < 0.1 and pos1 in ['CF', 'W']: xg = 0.45

        vec_att = np.array([[xg/1.1, p1['xA_p90']/0.8, p1['Prg_Sim']/60]])
        vec_def = np.array([[p2['Tkl_Sim']/8.0, p2['Int_Sim']/5.0, p2['Air_Sim']/6.0]])

        tahmin = model.predict([vec_att, vec_def], verbose=0)[0][0] * 100

        print(f"🧠 YAPAY ZEKA ANALİZİ: %{tahmin:.1f} Hücum Başarısı")

        if tahmin > 55: print(f"🔥 {p1['Player']} savunmayı geçer!")
        elif tahmin < 45: print(f"🧱 {p2['Player']} geçit vermez!")
        else: print("⚖️ Müthiş kapışma, ortada!")

    # === SENARYO C: TERS ===
    elif (pos1 in defans_grubu) and (pos2 in hucum_grubu):
        evrensel_analiz_detayli(oyuncu2_adi, oyuncu1_adi)
        return

    else:
        print("⚠️ Analiz yapılamadı.")

    print("=" * 60 + "\n")

# --- TESTLER ---

print("\n--- 💥 DÜELLOLAR (Icardi artık CF!) ---")
evrensel_analiz_detayli("Icardi", "Paulista")
evrensel_analiz_detayli("Jota", "Mert Müldür")

print("\n--- 🛡️ SAVUNMA KIYASLAMASI (Stoper vs Bek) ---")
evrensel_analiz_detayli("Oosterwolde", "Mustafa Eskihellaç")

print("\n--- ⚡ KANAT vs BEK (Kerem vs Gökhan) ---")
evrensel_analiz_detayli("Kerem Aktürkoğlu", "Gökhan Sazdağı")

✅ Oyuncular yeniden sınıflandırıldı (Icardi -> CF, Gökhan -> FB).

--- 💥 DÜELLOLAR (Icardi artık CF!) ---

🔍 Analiz: Icardi vs Paulista ...
------------------------------------------------------------
📢 MAURO ICARDI (CF)
   Rol: Fırsatçı Golcü
              VS
📢 GABRIEL PAULISTA (CB)
   Rol: Standart Stoper (Durdurucu)
------------------------------------------------------------
🧠 YAPAY ZEKA ANALİZİ: %6.3 Hücum Başarısı
🧱 Gabriel Paulista geçit vermez!


🔍 Analiz: Jota vs Mert Müldür ...
------------------------------------------------------------
📢 JOTA SILVA (W)
   Rol: Kanat Oyuncusu (Hücum)
              VS
📢 MERT MÜLDÜR (FB)
   Rol: Standart Bek (Destek)
------------------------------------------------------------
🧠 YAPAY ZEKA ANALİZİ: %83.1 Hücum Başarısı
🔥 Jota Silva savunmayı geçer!


--- 🛡️ SAVUNMA KIYASLAMASI (Stoper vs Bek) ---

🔍 Analiz: Oosterwolde vs Mustafa Eskihellaç ...
------------------------------------------------------------
📢 JAYDEN OOSTERWOLDE (CB)
   Rol: Pas D

In [6]:
# HÜCRE 7: SİSTEMİ PAKETLEME (UI İÇİN KAYIT)

import pickle

# 1. Modeli Kaydet (.h5 formatında)
model.save('futbol_analiz_modeli.h5')

# 2. Veri Setini Kaydet (Excel/CSV olarak)
# Web sitesi açıldığında bu temizlenmiş veriyi okuyacak
df_final.to_csv('super_lig_final_veri.csv', index=False)

# 3. Scaler'ları (Ölçekleyicileri) Kaydetmemiz lazım mı?
# Biz kodun içinde manuel bölme ( /1.2 vb.) yaptığımız için gerek yok.
# Mantığı kodun içine gömeceğiz.

print("✅ SİSTEM PAKETLENDİ!")
print("1. futbol_analiz_modeli.h5 (Yapay Zeka Beyni)")
print("2. super_lig_final_veri.csv (Oyuncu Veritabanı)")
print("Bu dosyalar şimdi sol taraftaki dosyalar panelinde oluştu.")

✅ SİSTEM PAKETLENDİ!
1. futbol_analiz_modeli.h5 (Yapay Zeka Beyni)
2. super_lig_final_veri.csv (Oyuncu Veritabanı)
Bu dosyalar şimdi sol taraftaki dosyalar panelinde oluştu.


In [7]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
import plotly.graph_objects as go

# --- SAYFA AYARLARI ---
st.set_page_config(page_title="Futbol AI Scout", page_icon="⚽", layout="wide")

# --- CSS ---
st.markdown("""
    <style>
    .stButton>button {
        width: 100%;
        background-color: #FF4B4B;
        color: white;
        font-weight: bold;
    }
    </style>
    """, unsafe_allow_html=True)

st.title("⚽ Yapay Zeka Destekli Futbol Analiz Sistemi")
st.markdown("**2025-2026 Süper Lig: Hücum vs Savunma Analizi**")

# --- 1. VERİ VE MODELİ YÜKLE ---
@st.cache_resource
def load_data_and_model():
    try:
        # Veriyi Oku
        df = pd.read_csv('super_lig_final_veri.csv')

        # --- 🚨 HATA DÜZELTİCİ YAMA (Pos_Simple Fix) 🚨 ---
        # Eğer Pos_Simple sütunu yoksa, Pos sütunundan yeniden üret
        if 'Pos_Simple' not in df.columns:
            if 'Pos' in df.columns:
                df['Pos_Simple'] = df['Pos'].astype(str).str.split(',').str[0]
            else:
                # Pos da yoksa varsayılan olarak MF ata (Kod patlamasın)
                df['Pos_Simple'] = 'MF'

        # xG sütun isimlerini garantiye al (xG_F veya xG_p90 olabilir)
        if 'xG_p90' not in df.columns and 'xG_F' in df.columns:
            df.rename(columns={'xG_F': 'xG_p90', 'xA_F': 'xA_p90'}, inplace=True)
        # ----------------------------------------------------

        # Modeli Oku
        model = tf.keras.models.load_model('futbol_analiz_modeli.h5')
        return df, model

    except Exception as e:
        st.error(f"⚠️ Kritik Hata: {e}")
        st.stop()
        return None, None

df, model = load_data_and_model()

if df is not None:

    # --- KENAR ÇUBUĞU ---
    st.sidebar.header("🕵️‍♂️ Oyuncu Seçimi")
    takimlar = sorted(df['Takim'].unique().astype(str))

    # Oyuncu 1
    st.sidebar.subheader("1. Oyuncu (Hücum)")
    t1 = st.sidebar.selectbox("Takım", takimlar, index=0, key='t1')
    o1_list = df[df['Takim'] == t1]['Player'].tolist()
    o1_name = st.sidebar.selectbox("Oyuncu", o1_list, key='o1')

    # Oyuncu 2
    st.sidebar.subheader("2. Oyuncu (Savunma)")
    t2 = st.sidebar.selectbox("Takım", takimlar, index=1, key='t2')
    o2_list = df[df['Takim'] == t2]['Player'].tolist()
    o2_name = st.sidebar.selectbox("Oyuncu", o2_list, key='o2')

    # Verileri Çek
    p1 = df[df['Player'] == o1_name].iloc[0]
    p2 = df[df['Player'] == o2_name].iloc[0]

    # --- OYUNCU KARTLARI ---
    c1, c2, c3 = st.columns([1, 0.2, 1])
    with c1:
        st.info(f"🔵 {p1['Player']}")
        # Hata veren satır burasıydı, artık Pos_Simple garantilendiği için çalışacak
        st.caption(f"{p1['Takim']} | {p1['Pos_Simple']} | {p1['Rol']}")
        col_a, col_b = st.columns(2)
        col_a.metric("xG (Gol Beklentisi)", f"{p1['xG_p90']:.2f}")
        col_b.metric("Topla Çıkış", f"{p1['Prg_Sim']:.1f}")

    with c2:
        st.markdown("<h2 style='text-align: center; margin-top: 20px;'>VS</h2>", unsafe_allow_html=True)

    with c3:
        st.error(f"🔴 {p2['Player']}")
        st.caption(f"{p2['Takim']} | {p2['Pos_Simple']} | {p2['Rol']}")
        col_a, col_b = st.columns(2)
        col_a.metric("Top Kapma", f"{p2['Tkl_Sim']:.2f}")
        col_b.metric("Hava Hakimiyeti", f"{p2['Air_Sim']:.2f}")

    # --- ANALİZ BUTONU ---
    if st.button("🔥 EŞLEŞMEYİ ANALİZ ET"):
        st.divider()

        # Veri Hazırlığı (Model için)
        # 0 xG Düzeltmesi
        xg = p1['xG_p90']
        if xg < 0.1 and ('FW' in str(p1['Pos_Simple']) or 'Forvet' in str(p1['Rol'])): xg = 0.45

        vec_att = np.array([[xg/1.1, p1['xA_p90']/0.8, p1['Prg_Sim']/60]])
        vec_def = np.array([[p2['Tkl_Sim']/8.0, p2['Int_Sim']/5.0, p2['Air_Sim']/6.0]])

        # Yapay Zeka Tahmini
        tahmin = model.predict([vec_att, vec_def], verbose=0)[0][0]
        yuzde = tahmin * 100

        # Sonuç Gösterimi
        st.subheader("🧠 Yapay Zeka Kararı")
        st.progress(int(yuzde))

        col_res1, col_res2 = st.columns([3, 1])

        with col_res1:
            if yuzde > 55:
                st.success(f"🔥 **{p1['Player']} AVANTAJLI!** (%{yuzde:.1f})")
                st.write("Yapay zeka, hücumcunun bitiricilik ve hızını savunmaya göre daha baskın buldu.")
            elif yuzde < 45:
                st.error(f"🧱 **{p2['Player']} DUVAR OLDU!** (%{100-yuzde:.1f})")
                st.write("Savunmacının fiziksel gücü ve müdahale yeteneği, golcüyü kilitliyor.")
            else:
                st.warning(f"⚖️ **ÇOK DENGELİ!** (%{yuzde:.1f})")
                st.write("Tam anlamıyla bir derbi eşleşmesi. Günlük performans belirleyici olur.")

        with col_res2:
            # Radar Grafiği (Görsel Şov)
            categories = ['Gol/xG', 'Asist', 'Hız/Dripling', 'Top Kapma', 'Pas Arası', 'Hava Topu']
            v1 = [p1['xG_p90']*20, p1['xA_p90']*20, p1['Prg_Sim']/2, p1['Tkl_Sim']*5, p1['Int_Sim']*5, p1['Air_Sim']*5]
            v2 = [p2['xG_p90']*20, p2['xA_p90']*20, p2['Prg_Sim']/2, p2['Tkl_Sim']*5, p2['Int_Sim']*5, p2['Air_Sim']*5]

            fig = go.Figure()
            fig.add_trace(go.Scatterpolar(r=v1, theta=categories, fill='toself', name=p1['Player']))
            fig.add_trace(go.Scatterpolar(r=v2, theta=categories, fill='toself', name=p2['Player']))
            fig.update_layout(polar=dict(radialaxis=dict(visible=False, range=[0, 30])), showlegend=False, margin=dict(t=0, b=0, l=0, r=0), height=250)
            st.plotly_chart(fig, use_container_width=True)

Writing app.py


In [8]:
# HÜCRE 8: Cloudflare Tüneli ile Web Sitesini Başlat (Şifresiz & Hızlı)

import time
import os
import subprocess
import re

print("🚀 Web Uygulaması Cloudflare ile Başlatılıyor...")

# 1. Önceki Streamlit süreçlerini temizle (Çakışmayı önle)
os.system("pkill -9 streamlit")

# 2. Streamlit'i Arka Planda Başlat
os.system("streamlit run app.py &>/dev/null&")
print("⏳ Streamlit sunucusu hazırlanıyor...")
time.sleep(3)

# 3. Cloudflare Tunnel'ı İndir (Eğer yoksa)
if not os.path.exists("cloudflared"):
    print("📥 Cloudflare aracı indiriliyor...")
    os.system("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x cloudflared")

# 4. Tüneli Başlat
# Logları 'cloudflare.log' dosyasına yazdırıyoruz ki linki oradan okuyabilelim
print("🚇 Tünel açılıyor...")
os.system("nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflare.log 2>&1 &")

# 5. Linki Bul ve Yazdır
time.sleep(5) # Tünelin oluşması için bekle

found_link = False
try:
    with open('cloudflare.log', 'r') as f:
        for line in f:
            # Linki Regex ile yakala (trycloudflare.com)
            match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
            if match:
                print("\n" + "="*60)
                print(f"🎉 SİTE HAZIR! AŞAĞIDAKİ LİNKE TIKLA:")
                print(f"👉 {match.group(0)}")
                print("="*60)
                found_link = True
                break

    if not found_link:
        print("⚠️ Link log dosyasında henüz bulunamadı. Lütfen 5-10 saniye bekleyip bu hücreyi tekrar çalıştır.")
        # Hata ayıklama için logun son satırlarını göster
        print("\n--- Log Dosyası Son Satırlar ---")
        os.system("tail -n 5 cloudflare.log")

except FileNotFoundError:
    print("❌ Log dosyası oluşturulamadı. Tünel başlatılamadı.")

🚀 Web Uygulaması Cloudflare ile Başlatılıyor...
⏳ Streamlit sunucusu hazırlanıyor...
📥 Cloudflare aracı indiriliyor...
🚇 Tünel açılıyor...

🎉 SİTE HAZIR! AŞAĞIDAKİ LİNKE TIKLA:
👉 https://commission-speeds-join-nurse.trycloudflare.com


In [9]:
import os
import time
import subprocess
import re

print("🔧 SİSTEM TAMİR EDİLİYOR VE BAŞLATILIYOR...")

# --- 1. ESKİ SÜREÇLERİ ÖLDÜR (TEMİZLİK) ---
os.system("pkill -9 streamlit")
os.system("pkill -9 cloudflared")
print("🧹 Eski süreçler temizlendi.")

# --- 2. APP.PY DOSYASINI YENİDEN YAZ (HATASIZ SÜRÜM) ---
# Bu kısım app.py dosyasını sıfırdan, düzeltilmiş haliyle oluşturur.
app_code = """
import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
import plotly.graph_objects as go
import os

# Sayfa Ayarları
st.set_page_config(page_title="Futbol AI Scout", page_icon="⚽", layout="wide")
st.markdown("<style>.stButton>button {width: 100%; background-color: #FF4B4B; color: white; font-weight: bold;}</style>", unsafe_allow_html=True)

st.title("⚽ Yapay Zeka Destekli Futbol Analiz Sistemi")
st.markdown("**2025-2026 Süper Lig: Hücum vs Savunma Analizi**")

# Veri Yükleme Fonksiyonu
@st.cache_resource
def load_data_and_model():
    try:
        # Veri Kontrolü
        if not os.path.exists('super_lig_final_veri.csv'):
            st.error("❌ Veri dosyası (super_lig_final_veri.csv) bulunamadı! Lütfen önceki adımları çalıştırdığından emin ol.")
            st.stop()

        df = pd.read_csv('super_lig_final_veri.csv')

        # --- HATA DÜZELTİCİ YAMALAR ---
        # 1. Pos_Simple Eksikse Oluştur
        if 'Pos_Simple' not in df.columns:
            if 'Pos' in df.columns:
                df['Pos_Simple'] = df['Pos'].astype(str).str.split(',').str[0]
            else:
                df['Pos_Simple'] = 'MF' # Varsayılan

        # 2. Sütun İsimlerini Garantiye Al
        if 'xG_p90' not in df.columns and 'xG_F' in df.columns:
            df.rename(columns={'xG_F': 'xG_p90', 'xA_F': 'xA_p90'}, inplace=True)

        # 3. Modeli Yükle
        if not os.path.exists('futbol_analiz_modeli.h5'):
            st.error("❌ Model dosyası (futbol_analiz_modeli.h5) bulunamadı!")
            st.stop()

        model = tf.keras.models.load_model('futbol_analiz_modeli.h5')
        return df, model

    except Exception as e:
        st.error(f"⚠️ Kritik Hata: {e}")
        st.stop()
        return None, None

df, model = load_data_and_model()

if df is not None:
    # Kenar Çubuğu
    st.sidebar.header("🕵️‍♂️ Oyuncu Seçimi")
    takimlar = sorted(df['Takim'].astype(str).unique())

    st.sidebar.subheader("1. Oyuncu (Hücum)")
    t1 = st.sidebar.selectbox("Takım", takimlar, index=0, key='t1')
    o1_list = df[df['Takim'] == t1]['Player'].tolist()
    o1_name = st.sidebar.selectbox("Oyuncu", o1_list, key='o1')

    st.sidebar.subheader("2. Oyuncu (Savunma)")
    t2 = st.sidebar.selectbox("Takım", takimlar, index=min(1, len(takimlar)-1), key='t2')
    o2_list = df[df['Takim'] == t2]['Player'].tolist()
    o2_name = st.sidebar.selectbox("Oyuncu", o2_list, key='o2')

    # Oyuncuları Seç
    p1 = df[df['Player'] == o1_name].iloc[0]
    p2 = df[df['Player'] == o2_name].iloc[0]

    # Kartlar
    c1, c2, c3 = st.columns([1, 0.2, 1])
    with c1:
        st.info(f"🔵 {p1['Player']}")
        st.caption(f"{p1['Takim']} | {p1['Pos_Simple']} | {p1['Rol']}")
        col_a, col_b = st.columns(2)
        col_a.metric("xG", f"{p1.get('xG_p90', 0):.2f}")
        col_b.metric("Topla Çıkış", f"{p1.get('Prg_Sim', 0):.1f}")

    with c2:
        st.markdown("<h2 style='text-align: center; margin-top: 20px;'>VS</h2>", unsafe_allow_html=True)

    with c3:
        st.error(f"🔴 {p2['Player']}")
        st.caption(f"{p2['Takim']} | {p2['Pos_Simple']} | {p2['Rol']}")
        col_a, col_b = st.columns(2)
        col_a.metric("Top Kapma", f"{p2.get('Tkl_Sim', 0):.2f}")
        col_b.metric("Hava Topu", f"{p2.get('Air_Sim', 0):.2f}")

    # Analiz Butonu
    if st.button("🔥 EŞLEŞMEYİ ANALİZ ET"):
        st.divider()

        # Veri Hazırlığı
        xg = p1.get('xG_p90', 0)
        if xg < 0.1 and ('FW' in str(p1.get('Pos_Simple', '')) or 'Forvet' in str(p1['Rol'])): xg = 0.45

        vec_att = np.array([[xg/1.1, p1.get('xA_p90', 0)/0.8, p1.get('Prg_Sim', 0)/60]])
        vec_def = np.array([[p2.get('Tkl_Sim', 0)/8.0, p2.get('Int_Sim', 0)/5.0, p2.get('Air_Sim', 0)/6.0]])

        # Yapay Zeka Tahmini
        try:
            tahmin = model.predict([vec_att, vec_def], verbose=0)[0][0]
            yuzde = tahmin * 100

            st.subheader("🧠 Yapay Zeka Kararı")
            st.progress(int(yuzde))

            col_res1, col_res2 = st.columns([3, 1])
            with col_res1:
                if yuzde > 55:
                    st.success(f"🔥 **{p1['Player']} AVANTAJLI!** (%{yuzde:.1f})")
                    st.write("Yapay zeka, hücumcunun bitiricilik ve hızını savunmaya göre daha baskın buldu.")
                elif yuzde < 45:
                    st.error(f"🧱 **{p2['Player']} DUVAR OLDU!** (%{100-yuzde:.1f})")
                    st.write("Savunmacının fiziksel gücü ve müdahale yeteneği, golcüyü kilitliyor.")
                else:
                    st.warning(f"⚖️ **ÇOK DENGELİ!** (%{yuzde:.1f})")
                    st.write("Tam anlamıyla bir derbi eşleşmesi. Günlük performans belirleyici olur.")

            with col_res2:
                # Radar Grafiği
                categories = ['Gol/xG', 'Asist', 'Hız', 'Top Kapma', 'Pas Arası', 'Hava Topu']
                v1 = [p1.get('xG_p90',0)*20, p1.get('xA_p90',0)*20, p1.get('Prg_Sim',0)/2, p1.get('Tkl_Sim',0)*5, p1.get('Int_Sim',0)*5, p1.get('Air_Sim',0)*5]
                v2 = [p2.get('xG_p90',0)*20, p2.get('xA_p90',0)*20, p2.get('Prg_Sim',0)/2, p2.get('Tkl_Sim',0)*5, p2.get('Int_Sim',0)*5, p2.get('Air_Sim',0)*5]

                fig = go.Figure()
                fig.add_trace(go.Scatterpolar(r=v1, theta=categories, fill='toself', name=p1['Player']))
                fig.add_trace(go.Scatterpolar(r=v2, theta=categories, fill='toself', name=p2['Player']))
                fig.update_layout(polar=dict(radialaxis=dict(visible=False, range=[0, 30])), showlegend=False, margin=dict(t=0, b=0, l=0, r=0), height=250)
                st.plotly_chart(fig, use_container_width=True)

        except Exception as e:
            st.error(f"Tahmin hatası: {e}")

"""

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py başarıyla yenilendi.")

# --- 3. KÜTÜPHANELERİ TAZELENİYOR ---
os.system("pip install streamlit plotly tensorflow pandas numpy > /dev/null 2>&1")
print("📚 Kütüphaneler kontrol edildi.")

# --- 4. STREAMLIT'İ BAŞLAT VE LOGLARI İZLE ---
print("🚀 Streamlit başlatılıyor...")
os.system("nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &")

print("⏳ Sunucunun açılması bekleniyor (10 saniye)...")
time.sleep(10)

# Logları kontrol et: Hata var mı?
with open("streamlit.log", "r") as f:
    logs = f.read()
    if "Traceback" in logs or "Error" in logs:
        print("\n⚠️⚠️⚠️ STREAMLIT HATASI TESPİT EDİLDİ! ⚠️⚠️⚠️")
        print("İşte hata mesajı:")
        print("-" * 40)
        print(logs)
        print("-" * 40)
        print("Lütfen bu hatayı bana kopyalayıp at.")
    else:
        print("✅ Streamlit sorunsuz çalışıyor.")

        # --- 5. TÜNELİ AÇ ---
        print("🚇 Tünel açılıyor...")
        if not os.path.exists("cloudflared"):
            os.system("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
            os.system("chmod +x cloudflared")

        os.system("nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflare.log 2>&1 &")
        time.sleep(5)

        # Linki Bul
        try:
            with open('cloudflare.log', 'r') as f:
                found = False
                for line in f:
                    match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
                    if match:
                        print("\n" + "="*60)
                        print(f"🎉 SİTE HAZIR! TIKLA VE GİR: {match.group(0)}")
                        print("="*60)
                        found = True
                        break
                if not found:
                    print("⚠️ Link henüz oluşmadı. Lütfen 5 saniye sonra tekrar dene.")
        except:
            print("❌ Log dosyası okunamadı.")

🔧 SİSTEM TAMİR EDİLİYOR VE BAŞLATILIYOR...
🧹 Eski süreçler temizlendi.
✅ app.py başarıyla yenilendi.
📚 Kütüphaneler kontrol edildi.
🚀 Streamlit başlatılıyor...
⏳ Sunucunun açılması bekleniyor (10 saniye)...
✅ Streamlit sorunsuz çalışıyor.
🚇 Tünel açılıyor...

🎉 SİTE HAZIR! TIKLA VE GİR: https://met-gen-toolkit-labels.trycloudflare.com


In [10]:
!curl ipv4.icanhazip.com

34.127.22.179
